In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

print("--- Starting Phase 1: Data Loading ---")
BASE_PATH = '/kaggle/input/datasets/sanhithreddy/ndb-ufes-oral-risk/NDB-UFES An oral cancer and leukoplakia dataset composed of histopathological images and patient data/' 

# FIX 1: Point exactly to the subfolder where the images live
IMAGE_DIR = os.path.join(BASE_PATH, 'patch', 'images-patches')
CSV_PATH = os.path.join(BASE_PATH, 'ndb-ufes.csv')

df = pd.read_csv(CSV_PATH)
print(f"Total initial records: {len(df)}")

In [ ]:
print(df.columns.tolist())

In [ ]:
# ==========================================
# PHASE 2 & 3: BULLETPROOF PREPROCESSING
# ==========================================
print("\n--- Starting Phase 2: Feature Engineering & Auto-Pathing ---")

# 1. Map the 'diagnosis' column to High vs Low Risk
def map_risk(diagnosis_value):
    d = str(diagnosis_value).lower()
    if 'carcinoma' in d or 'with dysplasia' in d:
        return 'high_risk'
    else:
        return 'low_risk'

df['risk_label'] = df['diagnosis'].apply(map_risk)

# 2. Hunt down the exact folder containing the images
print("Hunting for the actual image folder...")
actual_img_dir = None
for root, dirs, files in os.walk(BASE_PATH):
    # Check if a known image from the CSV exists in this folder
    if '0000.png' in files: 
        actual_img_dir = root
        break

if actual_img_dir:
    print(f"SUCCESS! Images found at: {actual_img_dir}")
else:
    print("CRITICAL ERROR: Could not find the image files. Check if they were uploaded correctly.")

# 3. Create absolute paths for every image so Keras cannot possibly miss them
df['absolute_path'] = df['path'].apply(lambda x: os.path.join(actual_img_dir, str(x)))

# Verify the very first file actually exists on the hard drive
test_path = df['absolute_path'].iloc[0]
print(f"Testing first file path: {test_path}")
print(f"Does file exist? {os.path.exists(test_path)}")

print("\n--- Starting Phase 3: Splitting & Generators ---")
# 4. Split the data
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['risk_label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['risk_label'], random_state=42)

# 5. Set up Image Generators
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, horizontal_flip=True, vertical_flip=True)
test_val_datagen = ImageDataGenerator(rescale=1./255)

# 6. Stream images (NOTICE: directory=None because we are using absolute paths now)
print("\nLoading Training Data...")
train_gen = train_datagen.flow_from_dataframe(
    train_df, directory=None, x_col='absolute_path', y_col='risk_label',
    target_size=(224, 224), batch_size=16, class_mode='binary'
)

print("Loading Validation Data...")
val_gen = test_val_datagen.flow_from_dataframe(
    val_df, directory=None, x_col='absolute_path', y_col='risk_label',
    target_size=(224, 224), batch_size=16, class_mode='binary', shuffle=False
)

print("Loading Testing Data...")
test_gen = test_val_datagen.flow_from_dataframe(
    test_df, directory=None, x_col='absolute_path', y_col='risk_label',
    target_size=(224, 224), batch_size=16, class_mode='binary', shuffle=False
)

In [ ]:
import os

print("--- DIAGNOSTIC CHECK ---")
# 1. Look inside the patch folder
try:
    files = os.listdir(IMAGE_DIR)
    print(f"Total items in folder: {len(files)}")
    print(f"First 10 items: {files[:10]}")
except Exception as e:
    print(f"Folder error: {e}")

# 2. Look at what the 'path' column contains
print("\nFirst 5 rows of the 'path' column:")
print(df['path'].head())

In [ ]:
# ==========================================
# PHASE 4: MODEL BUILDING & TRAINING
# ==========================================
print("\n--- Starting Phase 4: Model Building (ResNet50) ---")
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze base layers to keep their pre-trained knowledge

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x) # Prevents overfitting
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

# Compile using Recall to catch True Positives (crucial for cancer detection)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.Recall(name='recall')])

print("Beginning Training (This will take a few minutes)...")
# Train the model! Watch the epochs output here.
history = model.fit(train_gen, validation_data=val_gen, epochs=10)

# Save the model
model.save('/kaggle/working/oral_risk_classifier.h5')
print("\nModel saved successfully to /kaggle/working/oral_risk_classifier.h5!")

In [ ]:
# ==========================================
# PHASE 5: EVALUATION ON UNSEEN DATA
# ==========================================
print("\n--- Starting Phase 5: Evaluation ---")
y_pred_prob = model.predict(test_gen)
y_pred = (y_pred_prob > 0.5).astype(int)
y_true = test_gen.classes

class_names = list(test_gen.class_indices.keys())

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix on Unseen Test Data')
plt.ylabel('Actual Risk')
plt.xlabel('Predicted Risk')
plt.show()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import EfficientNetV2B0

# ==========================================
# PHASE 4: ADVANCED MODELING (EfficientNet + Fine-Tuning)
# ==========================================
print("\n--- Starting Phase 4: Advanced Model Building ---")

# 1. Automatically calculate Class Weights to fix the imbalance
train_labels = train_gen.classes
weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weight_dict = dict(enumerate(weights))
print(f"Applying Class Weights to penalize missed cancer cases: {class_weight_dict}")

# 2. Upgrade to EfficientNetV2
# EfficientNetV2 is often superior to ResNet for complex medical textures
base_model = EfficientNetV2B0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 3. FINE TUNING: Unfreeze the top layers
base_model.trainable = True
# Freeze all layers EXCEPT the top 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

# 4. Compile with a lower learning rate (1e-4) so we don't destroy the pre-trained weights
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), 
    loss='binary_crossentropy', 
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
)

print("Beginning Fine-Tuning (This will take a bit longer)...")
history = model.fit(
    train_gen, 
    validation_data=val_gen, 
    epochs=15, 
    class_weight=class_weight_dict # Injecting the penalty weights here!
)

model.save('/kaggle/working/oral_risk_classifier_v2.h5')
print("\nAdvanced Model saved to /kaggle/working/oral_risk_classifier_v2.h5")

# ==========================================
# PHASE 5: EVALUATION ON UNSEEN DATA
# ==========================================
print("\n--- Starting Phase 5: Evaluation ---")
y_pred_prob = model.predict(test_gen)
y_pred = (y_pred_prob > 0.5).astype(int)
y_true = test_gen.classes

class_names = list(test_gen.class_indices.keys())

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (Advanced Model)')
plt.ylabel('Actual Risk')
plt.xlabel('Predicted Risk')
plt.show()

In [ ]:
# ==========================================
# PHASE 6: SINGLE IMAGE INFERENCE 
# ==========================================
print("\n--- Starting Phase 6: Inference Test ---")
def predict_single_image(image_path, model_path='/kaggle/working/oral_risk_classifier.h5'):
    # Load model
    loaded_model = tf.keras.models.load_model(model_path)
    
    # Load and preprocess image
    img = load_img(image_path, target_size=(224, 224))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Predict
    prediction_prob = loaded_model.predict(img_array, verbose=0)[0][0]
    
    # Interpret
    high_risk_idx = train_gen.class_indices['high_risk']
    if round(prediction_prob) == high_risk_idx:
        return f"HIGH RISK (Confidence: {prediction_prob:.4f})"
    else:
        return f"LOW RISK (Confidence: {(1.0 - prediction_prob):.4f})"

# Grab a random image from our test set using the absolute path we created
sample_image_path = test_df.iloc[0]['absolute_path']

print(f"Testing real-world inference on image: {sample_image_path}")
result = predict_single_image(sample_image_path)
print(f"\n>> Final Output: {result} <<")